[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/00_what_is_mujoco.ipynb)

# 教程 0：MuJoCo 是什么？

**不需要任何物理仿真经验。**

本教程带你从零开始，理解 MuJoCo 能做什么、它为什么存在、该如何去想它——全部通过你能亲眼看到的动手示例来完成。

In [ ]:
!pip install -q mujoco mediapy

In [ ]:
import mujoco
import numpy as np
import mediapy as media
print(f"MuJoCo {mujoco.__version__} ready!")

---
## MuJoCo 解决什么问题？

假设你在造一个机器人。在真正造出物理机器人（昂贵、缓慢、容易摔坏）之前，你会想：

1. **测试**你的机器人设计到底能不能走／抓／飞
2. **训练**一个 AI 控制器，用上百万次试错
3. **预测**机器人和物体相互作用时会发生什么

MuJoCo 是一个**物理仿真器**——一个在力、重力和接触作用下计算物体如何运动的软件。它回答的问题是：*"如果我施加这个力，接下来会发生什么？"*

### 为什么偏偏选 MuJoCo？

| 特性 | 为什么重要 |
|---------|---------------|
| **快** | 比实时快 10-100 倍——训练机器人用几分钟，不是几年 |
| **接触精确** | 把触碰、滑动、弹跳处理得很真实 |
| **稳定** | 即使是复杂机器人也不会"炸开"或产出垃圾结果 |
| **GPU 并行（MJX）** | 在一块 GPU 上同时仿真 10,000 个机器人 |
| **免费且开源** | Apache 2.0 许可，由 Google DeepMind 维护 |

---
## 核心思路：描述 → 仿真 → 观察

使用 MuJoCo 永远遵循三个步骤：

```
1. DESCRIBE    →    2. SIMULATE    →    3. OBSERVE
   (what exists)       (advance time)       (read results)
```

我们用尽可能最简单的例子来看每一步：**一个在重力下下落的小球。**

### 第 1 步：描述（DESCRIBE）——告诉 MuJoCo 存在些什么

我们用 XML（一种叫 MJCF 的文本格式）写出描述。把它想成一份菜谱：

- "有一片平坦的地面"
- "在 2 米高度处有一个球"
- "重力以 9.81 m/s² 把东西往下拉"

In [ ]:
# This XML string describes our world
world_description = """
<mujoco>
  <option gravity="0 0 -9.81"/>          <!-- Gravity points down (negative Z) -->
  <worldbody>                              <!-- Everything physical goes here -->
    <light pos="0 0 3" dir="0 0 -1"/>    <!-- A light so we can see -->
    <geom type="plane" size="2 2 0.1" rgba=".8 .8 .8 1"/>  <!-- The ground -->
    <body pos="0 0 2">                    <!-- A body (movable thing) at height 2m -->
      <joint type="free"/>                <!-- It can move freely in all directions -->
      <geom type="sphere" size="0.15" rgba="1 0.3 0 1"/>  <!-- It looks like an orange ball -->
    </body>
  </worldbody>
</mujoco>
"""

# MuJoCo "compiles" the description into an efficient internal format
model = mujoco.MjModel.from_xml_string(world_description)
print("✓ World described and compiled!")
print(f"  - Bodies: {model.nbody} (includes the 'world' itself)")
print(f"  - Geoms (shapes): {model.ngeom}")
print(f"  - Gravity: {model.opt.gravity}")

### 第 2 步：仿真（SIMULATE）——推进时间

现在我们创建**状态**（某一时刻的位置、速度），然后让 MuJoCo 计算接下来会发生什么。

In [ ]:
# Create the initial state (ball at height 2, not moving)
data = mujoco.MjData(model)

print(f"At time {data.time:.3f}s: ball height = {data.qpos[2]:.3f}m")

# Advance simulation by one tiny timestep (0.002 seconds by default)
mujoco.mj_step(model, data)

print(f"At time {data.time:.3f}s: ball height = {data.qpos[2]:.6f}m")
print(f"  (It moved {2.0 - data.qpos[2]:.6f}m downward — gravity is working!)")

### 第 3 步：观察（OBSERVE）——看看发生了什么

我们仿真更长的时间，看着球下落并弹起：

In [ ]:
# Reset everything to the start
data = mujoco.MjData(model)

# Create a renderer (a virtual camera)
renderer = mujoco.Renderer(model, height=400, width=500)

# Simulate for 3 seconds, capturing frames for a video
frames = []
while data.time < 3.0:
    mujoco.mj_step(model, data)       # Physics step
    
    # Capture a frame every 1/30th of a second (30 fps video)
    if len(frames) < data.time * 30:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

renderer.close()
media.show_video(frames, fps=30)

**就这些！** 你刚刚做到了：
1. 描述了一个世界（XML）→ 编译成 `model`
2. 仿真它（在循环里调用 `mj_step`）→ 更新了 `data`
3. 观察结果（渲染出一帧帧画面）

MuJoCo 里其余的一切都是在这个模式上搭建起来的。

---
## 两个核心对象

MuJoCo 只有**两个**主要的数据结构。理解它们是理解一切的钥匙：

### `MjModel` —— 蓝图（不改变）

包含仿真过程中保持**恒定**的一切：
- 存在多少个物体、它们的形状和大小
- 质量、摩擦、刚度
- 关节类型与限位
- 重力、仿真步长设置

**类比：** 一栋建筑的图纸——它告诉你那里有什么，但不告诉你人现在在哪儿。

### `MjData` —— 快照（每一步都在变）

包含随时间**变化**的一切：
- 当前的位置（`data.qpos`）和速度（`data.qvel`）
- 当前的时间（`data.time`）
- 力、接触、传感器读数
- 控制信号（`data.ctrl`）

**类比：** 一张监控摄像头的抓拍——告诉你*此刻*所有东西都在哪里。

In [ ]:
# model = the blueprint (constant)
print("=== MODEL (blueprint) ===")
print(f"  Timestep: {model.opt.timestep}s (fixed)")
print(f"  Ball radius: {model.geom_size[1, 0]}m (fixed)")
print(f"  Gravity: {model.opt.gravity} (fixed)")

print()

# data = the current state (changes)
print("=== DATA (current snapshot) ===")
print(f"  Time: {data.time:.3f}s")
print(f"  Ball position: {data.qpos[:3]}")
print(f"  Ball velocity: {data.qvel[:3]}")

---
## 基本构件：刚体、关节、几何体

每个 MuJoCo 世界都由三种基本构件组成：

| 概念 | 它是什么 | 现实世界类比 |
|---------|-----------|-------------------|
| **刚体（Body）** | 一个可以运动的刚性坐标系 | 你骨架里的一块骨头 |
| **关节（Joint）** | 连接刚体，允许特定的运动 | 膝盖或手肘 |
| **几何体（Geom）** | 附着在刚体上的形状（用于碰撞与外观） | 骨头上的肉／皮肤 |

我们一步步搭出点东西，看看它们是如何协同工作的：

In [ ]:
# A pendulum: one body, connected to the world by a hinge joint
pendulum_xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 -1 2" dir="0 0.5 -1"/>
    
    <!-- The pendulum arm -->
    <body name="arm" pos="0 0 1.5">
      <!-- THIS is what makes it swing: a hinge joint -->
      <!-- Without this joint, the arm would be frozen in place -->
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      
      <!-- The visible/physical shape -->
      <geom type="capsule" fromto="0 0 0 0 0 -0.8" size="0.04" 
            mass="1" rgba="0.2 0.6 0.9 1"/>
    </body>
  </worldbody>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(pendulum_xml)
d = mujoco.MjData(m)

# Start the pendulum at 90 degrees
d.qpos[0] = np.pi / 2  # 90° in radians

# Simulate
renderer = mujoco.Renderer(m, height=400, width=400)
frames = []
while d.time < 5.0:
    mujoco.mj_step(m, d)
    if len(frames) < d.time * 30:
        renderer.update_scene(d)
        frames.append(renderer.render().copy())
renderer.close()

media.show_video(frames, fps=30)

**发生了什么：**
- 我们创建了一个**刚体（body）**（那条手臂）
- 我们给了它一个**旋转关节（hinge joint）**——这让它能绕一根轴转动（像门合页）
- 我们给了它一个**几何体（geom）**（胶囊形状）——于是它有了质量、也看得见
- 剩下的交给重力！

### 关节类型

| 关节类型 | 自由度 | 例子 |
|-----------|-------------------|--------|
| `free` | 6（任意移动 + 任意转动） | 无人机、被抛出的球 |
| `hinge` | 1（绕轴转动） | 门、手肘、轮子 |
| `slide` | 1（沿轴平移） | 抽屉、活塞 |
| `ball` | 3（朝任意方向转动） | 肩膀、髋部 |

---
## 加入控制：让东西按你的意图动起来

到目前为止，只有重力在推动东西。要**控制**一个机器人，我们需要**执行器（actuator）**——把力施加到关节上的马达。

In [ ]:
controlled_xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 -1 2" dir="0 0.5 -1"/>
    <body pos="0 0 1.5">
      <joint name="hinge" type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0 0 -0.8" size="0.04" mass="1" rgba="0.9 0.4 0.1 1"/>
    </body>
  </worldbody>
  
  <!-- NEW: A motor that can apply torque to the hinge -->
  <actuator>
    <motor joint="hinge" ctrlrange="-5 5"/>  <!-- ±5 Nm of torque -->
  </actuator>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(controlled_xml)
d = mujoco.MjData(m)

# Our goal: hold the pendulum at 45 degrees (π/4 radians)
target = np.pi / 4

renderer = mujoco.Renderer(m, height=400, width=400)
frames = []

while d.time < 6.0:
    # === THE CONTROLLER ===
    # Read current angle and speed
    current_angle = d.qpos[0]
    current_speed = d.qvel[0]
    
    # Compute the motor command (PD controller):
    #   - Push toward target (proportional)
    #   - Slow down when moving fast (damping)
    error = target - current_angle
    motor_command = 10.0 * error - 3.0 * current_speed
    
    # Send command to the motor (clipped to ±5 Nm)
    d.ctrl[0] = np.clip(motor_command, -5, 5)
    # === END CONTROLLER ===
    
    mujoco.mj_step(m, d)
    
    if len(frames) < d.time * 30:
        renderer.update_scene(d)
        frames.append(renderer.render().copy())

renderer.close()
media.show_video(frames, fps=30)
print(f"Final angle: {np.degrees(d.qpos[0]):.1f}° (target was {np.degrees(target):.1f}°)")

**控制回路的模式**——这就是 MuJoCo 里所有机器人控制的工作方式：

```python
while simulating:
    # 1. READ: Get current state from data
    angle = data.qpos[0]
    
    # 2. DECIDE: Compute what force to apply
    command = my_controller(angle, target)
    
    # 3. ACT: Set the control signal
    data.ctrl[0] = command
    
    # 4. STEP: Let physics happen
    mujoco.mj_step(model, data)
```

---
## 理解 `qpos` 和 `qvel`（广义坐标）

你会到处看到 `data.qpos` 和 `data.qvel`。直觉是这样的：

- **`qpos`** = 描述构型的所有位置／角度
- **`qvel`** = 所有的速度（每个位置变化得有多快）

对一个**旋转关节（hinge joint）**：`qpos` 有 1 个数（以弧度计的角度）  
对一个**自由关节（free joint）**：`qpos` 有 7 个数（x、y、z + 四元数旋转）

| 关节 | qpos 条目 | qvel 条目 |
|-------|-------------|-------------|
| hinge | 1（角度） | 1（角速度） |
| slide | 1（位置） | 1（线速度） |
| free | 7（位置 + 四元数） | 6（线速度 + 角速度） |
| ball | 4（四元数） | 3（角速度） |

⚠️ **注意：** 对 `free` 和 `ball` 关节，`qpos` 和 `qvel` 的大小*不同*！这是因为旋转需要 4 个数（四元数）来表示，却只需要 3 个数来描述变化率。

In [ ]:
# Example: a model with mixed joint types
mixed_xml = """
<mujoco>
  <worldbody>
    <body name="floating_box" pos="0 0 1">
      <joint type="free"/>       <!-- 7 qpos, 6 qvel -->
      <geom type="box" size="0.1 0.1 0.1"/>
    </body>
    <body name="hinged_arm" pos="1 0 1">
      <joint type="hinge" axis="0 1 0"/>  <!-- 1 qpos, 1 qvel -->
      <geom type="capsule" fromto="0 0 0 0 0 -0.5" size="0.02"/>
    </body>
  </worldbody>
</mujoco>
"""
m = mujoco.MjModel.from_xml_string(mixed_xml)
d = mujoco.MjData(m)

print(f"Total qpos size: {m.nq} (7 from free + 1 from hinge = 8)")
print(f"Total qvel size: {m.nv} (6 from free + 1 from hinge = 7)")
print(f"")
print(f"qpos = {d.qpos}")  # [x,y,z, qw,qx,qy,qz, hinge_angle]
print(f"qvel = {d.qvel}")  # [vx,vy,vz, wx,wy,wz, hinge_velocity]")

---
## 把一切合起来：一个迷你机器人

我们从零搭一条 2 连杆机械臂——把学过的东西全都用上：

In [ ]:
robot_xml = """
<mujoco>
  <option gravity="0 0 -9.81"/>
  <worldbody>
    <light pos="0 -1 2" dir="0 0.5 -1"/>
    <geom type="plane" size="1 1 0.1" rgba=".85 .85 .85 1"/>
    
    <!-- Base is fixed to the world -->
    <body name="upper_arm" pos="0 0 0.8">
      <joint name="shoulder" type="hinge" axis="0 1 0"/>     <!-- Shoulder -->
      <geom type="capsule" fromto="0 0 0 0.3 0 0" size="0.03" rgba="0.2 0.5 0.8 1"/>
      
      <body name="forearm" pos="0.3 0 0">
        <joint name="elbow" type="hinge" axis="0 1 0"/>     <!-- Elbow -->
        <geom type="capsule" fromto="0 0 0 0.25 0 0" size="0.025" rgba="0.8 0.4 0.2 1"/>
        <site name="hand" pos="0.25 0 0" size="0.02" rgba="1 0 0 1"/>  <!-- Fingertip -->
      </body>
    </body>
  </worldbody>
  
  <actuator>
    <motor name="shoulder_motor" joint="shoulder" ctrlrange="-10 10"/>
    <motor name="elbow_motor" joint="elbow" ctrlrange="-5 5"/>
  </actuator>
</mujoco>
"""

m = mujoco.MjModel.from_xml_string(robot_xml)
d = mujoco.MjData(m)

print(f"Our robot has:")
print(f"  {m.njnt} joints (shoulder + elbow)")
print(f"  {m.nu} actuators (one motor per joint)")
print(f"  {m.nq} degrees of freedom")

# Make the arm trace a circle!
renderer = mujoco.Renderer(m, height=400, width=500)
frames = []

while d.time < 8.0:
    # Move joints in a sinusoidal pattern
    t = d.time
    target_shoulder = 0.5 * np.sin(t * 1.5)
    target_elbow = 0.8 * np.sin(t * 2.0 + 1.0)
    
    # PD control for both joints
    d.ctrl[0] = 20 * (target_shoulder - d.qpos[0]) - 5 * d.qvel[0]
    d.ctrl[1] = 10 * (target_elbow - d.qpos[1]) - 3 * d.qvel[1]
    
    mujoco.mj_step(m, d)
    if len(frames) < d.time * 30:
        renderer.update_scene(d)
        frames.append(renderer.render().copy())

renderer.close()
media.show_video(frames, fps=30)

---
## 小结：MuJoCo 的心智模型

```
┌─────────────────────────────────────────────────────────┐
│                    YOUR CODE                             │
│                                                         │
│  1. Write XML (once)  ──►  model = MjModel(xml)        │
│                                    │                    │
│  2. Create state      ──►  data = MjData(model)        │
│                                    │                    │
│  3. Loop:                          ▼                    │
│     ┌──────────────────────────────────────┐            │
│     │  Read: angle = data.qpos[0]          │            │
│     │  Decide: command = controller(angle)  │            │
│     │  Act: data.ctrl[0] = command          │            │
│     │  Step: mj_step(model, data)           │ ◄── repeat│
│     │  (Optional) Render: renderer.render() │            │
│     └──────────────────────────────────────┘            │
└─────────────────────────────────────────────────────────┘
```

### 关键术语

| 术语 | 含义 |
|------|--------|
| **MJCF** | 用来描述模型的 XML 格式 |
| **MjModel** | 编译后的模型（恒定的蓝图） |
| **MjData** | 仿真状态（每一步都在变） |
| **qpos** | 广义位置（所有关节的角度／位置） |
| **qvel** | 广义速度 |
| **ctrl** | 发送给执行器的控制信号 |
| **mj_step** | 把物理推进一个仿真步长 |
| **mj_forward** | 不推进时间地重新计算派生量 |

---
## 接下来是什么？

现在你已经理解了基本原理，继续往下：

1. **[教程 01：基础](01_basics.ipynb)** —— 按名字访问、检查模型
2. **[教程 02：渲染](02_rendering.ipynb)** —— 相机、深度、分割
3. **[教程 03：仿真](03_simulation.ipynb)** —— 仿真步长、能量、积分器
4. **[教程 04：控制](04_control.ipynb)** —— PD 控制、伺服、增益调优

或者直接跳到具体主题：
- 用 XML 造机器人 → [教程 06](06_xml_modeling.ipynb)
- 强化学习 → [教程 08](08_gymnasium.ipynb)
- GPU 并行仿真 → [教程 07](07_mjx.ipynb)